# Real AI Models — Try it in PyTorch

This is an **optional** hands-on companion to [Chapter 10](https://learnai.robennals.org/real-ai). New to PyTorch? Start with the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch).

The chapter said that the models at the frontier publish their blueprints, and that they're the transformer you already know plus some tweaks. This notebook makes that literal. You will:

1. **Read the blueprints** of the biggest models in the world — including a one-trillion-parameter model — without downloading any of their weights.
2. **Run a real model** from a frontier family, right here.
3. **Run one on your own computer**, if you want to.

Part 1 needs no GPU and downloads only a few kilobytes. Part 2 downloads about 1.5 GB.

## Setup

Two libraries: `transformers` (Hugging Face's model library) and `huggingface_hub` (which just fetches files from their servers). Hugging Face is the site where most open-weight models are published.

This first cell prints what your machine has. Nothing here matters yet, but the answers change what Part 2 can do.

In [ ]:
import json
import torch
import transformers
from huggingface_hub import HfApi, hf_hub_download

print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")

if torch.cuda.is_available():
    print(f"GPU          {torch.cuda.get_device_name(0)}")
else:
    print("GPU          none — everything below still works, Part 2 is just slower")

## Part 1 — Read the blueprints

Every open-weight model on Hugging Face ships a file called `config.json`. It is the model's spec sheet: how many layers, how wide, how attention is arranged, how many experts.

Here is the important part: **that file is a few kilobytes**, completely separate from the weights. DeepSeek-V3's weights are hundreds of gigabytes. Its `config.json` is smaller than a photo. So you can read the design of a model you could never run.

Let's fetch one.

In [ ]:
path = hf_hub_download("deepseek-ai/DeepSeek-V3", "config.json")
config = json.load(open(path))

import os
print(f"File size: {os.path.getsize(path)} bytes\n")

for key in ["num_hidden_layers", "hidden_size", "num_attention_heads", "vocab_size"]:
    print(f"{key:24} {config[key]}")

Sixty-one layers, each 7168 numbers wide, 128 attention heads, a vocabulary of 129,280 tokens. Those four lines describe the same machine you built in the transformer chapter, just much bigger.

### One wrinkle before we go further

Newer models wrap their language settings inside a `text_config` section, because the model also handles images. So `config["num_hidden_layers"]` works for DeepSeek and crashes for Kimi K2.5.

One tiny helper fixes it everywhere.

In [ ]:
def spec(cfg):
    # Return the language-model part of a config, wherever it lives.
    return cfg.get("text_config", cfg)


def load_config(repo):
    return json.load(open(hf_hub_download(repo, "config.json")))


kimi = load_config("moonshotai/Kimi-K2.5")

print("Without the helper:", "num_hidden_layers" in kimi)
print("With the helper:   ", "num_hidden_layers" in spec(kimi))
print("\nKimi K2.5 layers:", spec(kimi)["num_hidden_layers"])

### How big is it, really?

You can also ask how many parameters a model has without downloading it. Hugging Face reads that straight from the headers of the weight files and reports it.

In [ ]:
api = HfApi()

GIANTS = [
    "Qwen/Qwen3-0.6B",
    "openai/gpt-oss-120b",
    "deepseek-ai/DeepSeek-V3",
    "zai-org/GLM-5.2",
    "moonshotai/Kimi-K2.5",
]

for repo in GIANTS:
    info = api.model_info(repo)
    total = getattr(info.safetensors, "total", None) if info.safetensors else None
    if total:
        print(f"{repo:34} {total/1e9:10.1f} billion parameters")
    else:
        print(f"{repo:34} {'not reported':>10}")

The last one is over a trillion parameters. You just measured it by downloading nothing.

### Where the experts are

The chapter said a mixture-of-experts model replaces one thinking layer with many, and wakes only a few per word. That shows up directly in the config.

Annoyingly, three labs picked three different names for the same idea. That's worth seeing, because it's a good reminder that there's no standards committee here — just labs copying each other's good ideas and renaming things along the way.

In [ ]:
# The same concept, three different key names.
EXPERT_KEYS = ["n_routed_experts", "num_local_experts", "num_experts"]

for repo in ["deepseek-ai/DeepSeek-V3", "openai/gpt-oss-120b", "zai-org/GLM-5.2"]:
    s = spec(load_config(repo))
    total_experts = next((s[k] for k in EXPERT_KEYS if k in s), None)
    awake = s.get("num_experts_per_tok")

    if total_experts and awake:
        used_key = next(k for k in EXPERT_KEYS if k in s)
        print(f"{repo}")
        print(f"   key used:   {used_key}")
        print(f"   experts:    {total_experts}, of which {awake} wake up per word")
        print(f"   that is:    {100 * awake / total_experts:.1f}% of the experts\n")

### A model that admits whose design it borrowed

Every config has an `architectures` field naming the Python class that runs the model. Look at what Moonshot's trillion-parameter Kimi says, reading inside `text_config` with the helper from earlier.

In [ ]:
for repo in ["deepseek-ai/DeepSeek-V3", "moonshotai/Kimi-K2.5"]:
    cfg = load_config(repo)
    # Use the helper: Kimi wraps its language settings inside text_config.
    print(f"{repo:34} {spec(cfg)['architectures']}")

Kimi K2.5 is a trillion-parameter model built by a different company in a different city, and it runs on the class named `DeepseekV3ForCausalLM`. It is, architecturally, a DeepSeek-V3.

This is the chapter's whole argument in one line of JSON. These labs are not each inventing a secret machine. They are building the same machine.

### The whole comparison

Let's put it together.

In [ ]:
def summarise(repo):
    s = spec(load_config(repo))
    info = api.model_info(repo)
    total = getattr(info.safetensors, "total", None) if info.safetensors else None
    experts = next((s[k] for k in EXPERT_KEYS if k in s), None)
    # transformers v5 moved rope_theta into a sub-dictionary
    rope = s.get("rope_theta") or s.get("rope_parameters", {}).get("rope_theta")
    return {
        "params": f"{total/1e9:.1f}B" if total else "?",
        "layers": s.get("num_hidden_layers", "?"),
        "width": s.get("hidden_size", "?"),
        "experts": experts or "none (dense)",
        "context": f"{s.get('max_position_embeddings', 0)/1000:.0f}k",
        "rope": rope or "?",
    }


rows = {r: summarise(r) for r in GIANTS}
cols = ["params", "layers", "width", "experts", "context", "rope"]

print(f"{'model':<34}" + "".join(f"{c:>14}" for c in cols))
for repo, r in rows.items():
    print(f"{repo:<34}" + "".join(f"{str(r[c]):>14}" for c in cols))

Every one of these uses RoPE for positions (that's the `rope` column — the number is a setting inside it). Every one but the smallest uses experts. They differ in size, not in kind.

## Part 2 — Run a real one

Reading blueprints is one thing. Let's actually run a model.

**Qwen3-0.6B** is the smallest member of the Qwen3 family. Its big sibling, Qwen3-235B, is a frontier model. This one has the same design — same rotary positions, same normalisation, same gated thinking layer — shrunk to 0.6 billion parameters so it fits anywhere. It is a real frontier-family model, not a toy.

The next cell downloads about 1.5 GB, so give it a minute.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3-0.6B"
on_gpu = torch.cuda.is_available()
device = "cuda" if on_gpu else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

# Older GPUs (like Colab's free T4) have no fast bfloat16, so ask for float16 there.
# `dtype` is the modern argument name; older transformers versions call it `torch_dtype`.
want = torch.float16 if on_gpu else torch.float32
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=want)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=want)

model = model.to(device)
model.eval()

print(f"Loaded {MODEL} on {device.upper()}")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f} million")

You may have noticed this model reports fewer parameters than the table above gave it. Hugging Face counts the input word-table and the output word-table separately, while the loaded model shares one between them. Same model, two reasonable ways to count.

Now ask it something. Qwen3 can "think" out loud before answering, which is interesting but slow, so we switch that off with `enable_thinking=False`.

In [ ]:
def ask(question, max_new_tokens=80):
    messages = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    reply = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True)


print(ask("In one sentence, why is the sky blue?"))

That is a real language model, running on your machine, answering a question you wrote.

Try your own questions below. Then try to find its edges — it is 0.6 billion parameters, roughly a four-hundredth the size of the models in the table above, and it will show. Ask it something obscure, or some arithmetic, and watch it fail in ways the big ones wouldn't.

In [ ]:
print(ask("What is 17 times 24?"))
print()
print(ask("Name a country whose flag is only red and white."))

Don't be surprised if one of those answers is confidently wrong. A 0.6-billion-parameter model will happily state a false fact in a perfectly reasonable tone, and it has no way of signalling that it isn't sure.

That is worth sitting with. It is the same architecture as the frontier models in the table above, trained the same way. The difference is size and training, and this is what the bottom of that range looks like.

## Part 3 — On your own computer

Everything above runs the same way on your own machine. Copy this notebook, install `torch` and `transformers`, and Part 1 works instantly, since it only downloads text files.

For running models locally, the easiest tool in 2026 is **[Ollama](https://ollama.com)**. It downloads a *quantized* model (the chapter's word for storing each number in fewer bits) and runs it with one command, on Mac, Windows, or Linux:

```bash
ollama run qwen3:0.6b
```

That is the same model you just ran, squeezed to around 400 MB. You can go much bigger:

```bash
ollama run qwen3:8b        # a serious model, needs ~8GB of memory
```

### How big a model will your computer take?

A rough rule: **memory needed ≈ parameters × bytes per number × 1.2.** Quantized to 4 bits that's about half a byte each; at full 16-bit precision it's two bytes each.

| Your memory | Quantized (4-bit) | Full precision (16-bit) |
|---|---|---|
| 8 GB | up to about 12B | up to about 3B |
| 16 GB | up to about 24B | up to about 7B |
| 32 GB | up to about 50B | up to about 14B |

Those are for short conversations. Long ones need extra room for the KV cache, which grows with every word.

Notice what this means for the table in Part 1. A trillion-parameter model needs hundreds of gigabytes. You are not running Kimi K2.5 at home. But you *can* read exactly how it was built, and you can run a model from the same family, built on the same design. That gap between reading and running is the honest state of things: the blueprints are public, the machines to execute them are not.

## What you did

- Read the spec sheets of models up to a trillion parameters, downloading a few kilobytes.
- Found the same design showing up under different names across rival labs, and one model whose config openly names another lab's architecture.
- Measured how few experts wake up for a single word.
- Ran a real frontier-family model and found its limits.

The chapters ahead take each of these apart: [mixture of experts](https://learnai.robennals.org/mixture-of-experts), [long context](https://learnai.robennals.org/long-context), and [running models fast](https://learnai.robennals.org/inference).